In [ ]:
#| default_exp registry

In [ ]:
#| export
from __future__ import annotations
import pkgutil
import inspect
import importlib
import kreview.features
from kreview.eval_engine import FeatureEvaluator
import structlog

log = structlog.get_logger()


In [ ]:
#| export
def get_all_evaluators() -> list[FeatureEvaluator]:
    """Dynamically discovers all FeatureEvaluator subclasses in kreview.features"""
    evaluators = []
    for _, module_name, _ in pkgutil.iter_modules(kreview.features.__path__):
        try:
            mod = importlib.import_module(f"kreview.features.{module_name}")
            for name, obj in inspect.getmembers(mod, inspect.isclass):
                if issubclass(obj, FeatureEvaluator) and obj != FeatureEvaluator:
                    if not any(isinstance(e, obj) for e in evaluators): # prevent duplicates
                        evaluators.append(obj())
        except Exception as e:
            log.warning("evaluator_import_failed", module=module_name, error=str(e))
    return evaluators


In [ ]:
#| test
evans = get_all_evaluators()
assert len(evans) > 0
